<a href="https://colab.research.google.com/github/naveendevatwork-spec/ICT-assignments/blob/main/Income_Prediction_Using_TensorFlow_and_PyTorch_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [62]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix,precision_score, recall_score, f1_score



In [29]:
df=pd.read_csv("/content/drive/MyDrive/ICT/AI ML/Datasets/adult_dataset.csv")
df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39.0,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40.0,NaN,<=50K
1,50.0,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13.0,United-States,<=50K
2,38.0,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40.0,United-States,<=50K
3,53.0,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40.0,United-States,<=50K
4,28.0,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40.0,Cuba,<=50K


In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             46400 non-null  float64
 1   workclass       45480 non-null  object 
 2   fnlwgt          48842 non-null  int64  
 3   education       48842 non-null  object 
 4   education-num   48842 non-null  int64  
 5   marital-status  48842 non-null  object 
 6   occupation      45485 non-null  object 
 7   relationship    48842 non-null  object 
 8   race            48842 non-null  object 
 9   sex             48842 non-null  object 
 10  capital-gain    48842 non-null  int64  
 11  capital-loss    48842 non-null  int64  
 12  hours-per-week  46400 non-null  float64
 13  native-country  46137 non-null  object 
 14  income          48842 non-null  object 
dtypes: float64(2), int64(4), object(9)
memory usage: 5.6+ MB


In [31]:
df.isna().sum()

,0
age,2442
workclass,3362
fnlwgt,0
education,0
education-num,0
marital-status,0
occupation,3357
relationship,0
race,0
sex,0


In [32]:
df.isna().sum()/len(df)*100

,0
age,4.999795
workclass,6.883420
fnlwgt,0.000000
education,0.000000
education-num,0.000000
marital-status,0.000000
occupation,6.873183
relationship,0.000000
race,0.000000
sex,0.000000


In [33]:
# Check duplicate rows
print("Number of duplicate rows:", df.duplicated().sum())

Number of duplicate rows: 16


In [34]:
# Remove duplicate rows
df = df.drop_duplicates()

# Verify duplicates are removed
print("duplicate rows:", df.duplicated().sum())

duplicate rows: 0


In [35]:
print("Question marks in each column:")
print((df == "?").sum())

Question marks in each column:
age                  0
workclass         1763
fnlwgt               0
education            0
education-num        0
marital-status       0
occupation        1758
relationship         0
race                 0
sex                  0
capital-gain         0
capital-loss         0
hours-per-week       0
native-country     555
income               0
dtype: int64


In [36]:
df.replace("?", np.nan, inplace=True)

In [37]:
print(df.isnull().sum())

age               2442
workclass         5125
fnlwgt               0
education            0
education-num        0
marital-status       0
occupation        5115
relationship         0
race                 0
sex                  0
capital-gain         0
capital-loss         0
hours-per-week    2442
native-country    3260
income               0
dtype: int64


In [38]:
print("Numerical columns:")
print(df.select_dtypes(include=np.number).columns)

print("\nCategorical columns:")
print(df.select_dtypes(include="object").columns)

Numerical columns:
Index(['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss',
       'hours-per-week'],
      dtype='object')

Categorical columns:
Index(['workclass', 'education', 'marital-status', 'occupation',
       'relationship', 'race', 'sex', 'native-country', 'income'],
      dtype='object')


In [39]:
from sklearn.impute import SimpleImputer

# Identify numerical and categorical columns
numerical_cols = df.select_dtypes(include=np.number).columns
categorical_cols = df.select_dtypes(include="object").columns

# Fill missing numerical values using median
num_imputer = SimpleImputer(strategy="median")
df[numerical_cols] = num_imputer.fit_transform(df[numerical_cols])

# Fill missing categorical values using mode
cat_imputer = SimpleImputer(strategy="most_frequent")
df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])

# Check missing values
print(df.isnull().sum())

age               0
workclass         0
fnlwgt            0
education         0
education-num     0
marital-status    0
occupation        0
relationship      0
race              0
sex               0
capital-gain      0
capital-loss      0
hours-per-week    0
native-country    0
income            0
dtype: int64


In [40]:
# Clean the income target labels
df['income'] = df['income'].str.strip().str.rstrip('.')

# Check the cleaned labels
print(df['income'].value_counts())

income
<=50K    37139
>50K     11687
Name: count, dtype: int64


#Train-Test split

In [41]:
# Separate features and target
X = df.drop('income', axis=1)
y = df['income']

#Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [42]:
# Identify categorical and numerical columns
categorical_cols = X_train.select_dtypes(include=['object']).columns
numerical_cols = X_train.select_dtypes(include=np.number).columns

# Create the encoder
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit on training data and transform both datasets
X_train_cat = encoder.fit_transform(X_train[categorical_cols])
X_test_cat = encoder.transform(X_test[categorical_cols])

# Keep numerical features
X_train_num = X_train[numerical_cols].to_numpy()
X_test_num = X_test[numerical_cols].to_numpy()

# Combine numerical and encoded categorical features
X_train_encoded = np.concatenate([X_train_num, X_train_cat], axis=1)
X_test_encoded = np.concatenate([X_test_num, X_test_cat], axis=1)

In [43]:
# Convert income labels into numerical values
y_train_encoded = y_train.map({'<=50K': 0, '>50K': 1})
y_test_encoded = y_test.map({'<=50K': 0, '>50K': 1})


In [44]:
scaler = StandardScaler()

# Scale numerical features using training data
X_train_num_scaled = scaler.fit_transform(X_train_num)

# Scale testing numerical features using the same scaler
X_test_num_scaled = scaler.transform(X_test_num)

# Combine scaled numerical features with encoded categorical features
X_train_final = np.concatenate(
    [X_train_num_scaled, X_train_cat], axis=1
)

X_test_final = np.concatenate(
    [X_test_num_scaled, X_test_cat], axis=1
)


#Tensorflow

In [49]:

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [50]:
# Automatically detect the number of input features
input_dim = X_train_final.shape[1]

model = Sequential([
    Dense(64, activation='relu', input_shape=(input_dim,)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

print("Number of input features:", input_dim)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                 │ (None, 64)             │         6,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,897 (34.75 KB)

 Trainable params: 8,897 (34.75 KB)

 Non-trainable params: 0 (0.00 B)

Number of input features: 105


In [51]:
history = model.fit(
    X_train_final,
    y_train_encoded,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/10
977/977 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8414 - loss: 0.3365 - val_accuracy: 0.8554 - val_loss: 0.3115
Epoch 2/10
977/977 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8549 - loss: 0.3114 - val_accuracy: 0.8564 - val_loss: 0.3094
Epoch 3/10
977/977 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8561 - loss: 0.3072 - val_accuracy: 0.8564 - val_loss: 0.3104
Epoch 4/10
977/977 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8586 - loss: 0.3040 - val_accuracy: 0.8565 - val_loss: 0.3090
Epoch 5/10
977/977 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8582 - loss: 0.3016 - val_accuracy: 0.8573 - val_loss: 0.3093
Epoch 6/10
977/977 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8611 - loss: 0.2978 - val_accuracy: 0.8569 - val_loss: 0.3113
Epoch 7/10
977/977 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.8626 - loss: 0.2956 - val_accuracy: 0.8579 - val_loss: 0.3116
Epoch 8/10
977/977 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.8619 - loss: 0.2934 - val_accuracy: 0.

In [52]:
y_pred_prob = model.predict(X_test_final)
y_pred = (y_pred_prob > 0.5).astype(int).ravel()

print("Accuracy:", accuracy_score(y_test_encoded, y_pred))
print("\nClassification Report:\n", classification_report(y_test_encoded, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test_encoded, y_pred))

306/306 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Accuracy: 0.8561335244726602

Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.94      0.91      7428
           1       0.75      0.60      0.67      2338

    accuracy                           0.86      9766
   macro avg       0.82      0.77      0.79      9766
weighted avg       0.85      0.86      0.85      9766


Confusion Matrix:
 [[6960  468]
 [ 937 1401]]


#Pytorch

In [53]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


In [58]:
X_train_tensor = torch.tensor(X_train_final, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_final, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train_encoded.to_numpy(), dtype=torch.long)
y_test_tensor = torch.tensor(y_test_encoded.to_numpy(), dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

print("Training data:", X_train_tensor.shape)
print("Testing data:", X_test_tensor.shape)

Training data: torch.Size([39060, 105])
Testing data: torch.Size([9766, 105])


In [59]:
model_pytorch = nn.Sequential(
    nn.Linear(105, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 2)
)

print(model_pytorch)

Sequential(
  (0): Linear(in_features=105, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=2, bias=True)
)


In [60]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_pytorch.parameters(), lr=0.001)

In [61]:
for epoch in range(10):
    for X_batch, y_batch in train_loader:
        outputs = model_pytorch(X_batch)
        loss = criterion(outputs, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch [{epoch+1}/10], Loss: {loss.item():.4f}")

Epoch [1/10], Loss: 0.2279
Epoch [2/10], Loss: 0.4598
Epoch [3/10], Loss: 0.2793
Epoch [4/10], Loss: 0.5004
Epoch [5/10], Loss: 0.2484
Epoch [6/10], Loss: 0.2357
Epoch [7/10], Loss: 0.3044
Epoch [8/10], Loss: 0.2451
Epoch [9/10], Loss: 0.3651
Epoch [10/10], Loss: 0.3848


In [63]:
model_pytorch.eval()

with torch.no_grad():
    outputs = model_pytorch(X_test_tensor)
    y_pred_pytorch = torch.argmax(outputs, dim=1).numpy()

print("Accuracy:", accuracy_score(y_test_encoded, y_pred_pytorch))
print("\nClassification Report:\n", classification_report(y_test_encoded, y_pred_pytorch))
print("\nConfusion Matrix:\n", confusion_matrix(y_test_encoded, y_pred_pytorch))

Accuracy: 0.8571574851525702

Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.94      0.91      7428
           1       0.75      0.61      0.67      2338

    accuracy                           0.86      9766
   macro avg       0.82      0.77      0.79      9766
weighted avg       0.85      0.86      0.85      9766


Confusion Matrix:
 [[6949  479]
 [ 916 1422]]


In [64]:
results = pd.DataFrame({
    'Model': ['TensorFlow', 'PyTorch'],
    'Accuracy': [
        accuracy_score(y_test_encoded, y_pred),
        accuracy_score(y_test_encoded, y_pred_pytorch)
    ],
    'Precision': [
        precision_score(y_test_encoded, y_pred),
        precision_score(y_test_encoded, y_pred_pytorch)
    ],
    'Recall': [
        recall_score(y_test_encoded, y_pred),
        recall_score(y_test_encoded, y_pred_pytorch)
    ],
    'F1-Score': [
        f1_score(y_test_encoded, y_pred),
        f1_score(y_test_encoded, y_pred_pytorch)
    ]
})

results

,Model,Accuracy,Precision,Recall,F1-Score
0,TensorFlow,0.856134,0.749599,0.599230,0.666033
1,PyTorch,0.857157,0.748027,0.608212,0.670913


In [65]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.4/270.4 kB 21.8 MB/s eta 0:00:00


In [66]:
import optuna

In [67]:
X_train_opt, X_val_opt, y_train_opt, y_val_opt = train_test_split(
    X_train_final,
    y_train_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_train_encoded
)

X_train_opt = torch.tensor(X_train_opt, dtype=torch.float32)
X_val_opt = torch.tensor(X_val_opt, dtype=torch.float32)

y_train_opt = torch.tensor(y_train_opt.to_numpy(), dtype=torch.long)
y_val_opt = torch.tensor(y_val_opt.to_numpy(), dtype=torch.long)

print("Training data:", X_train_opt.shape)
print("Validation data:", X_val_opt.shape)

Training data: torch.Size([31248, 105])
Validation data: torch.Size([7812, 105])


In [70]:
def objective(trial):
    hidden1 = trial.suggest_categorical('hidden1', [32, 64, 128])
    hidden2 = trial.suggest_categorical('hidden2', [16, 32, 64])

    learning_rate = trial.suggest_float(
        'learning_rate', 0.0001, 0.003, log=True
    )

    model = nn.Sequential(
        nn.Linear(105, hidden1),
        nn.ReLU(),
        nn.Linear(hidden1, hidden2),
        nn.ReLU(),
        nn.Linear(hidden2, 2)
    )

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    train_dataset = TensorDataset(
        X_train_opt, y_train_opt
    )

    train_loader_opt = DataLoader(
        train_dataset,
        batch_size=32,
        shuffle=True
    )

    for epoch in range(10):
        model.train()

        for X_batch, y_batch in train_loader_opt:
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    model.eval()

    with torch.no_grad():
        outputs = model(X_val_opt)
        predictions = torch.argmax(outputs, dim=1)

        accuracy = (
            predictions == y_val_opt
        ).float().mean().item()

    return accuracy

In [71]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=5)

print("Best hyperparameters:", study.best_params)
print("Best validation accuracy:", study.best_value)

[I 2026-10-22 17:28:56,163] A new study created in memory with name: no-name-d6e6be10-89c2-4bf0-a130-f2fb7171e614
[I 2026-10-22 17:29:19,330] Trial 0 finished with value: 0.8572708368301392 and parameters: {'hidden1': 64, 'hidden2': 64, 'learning_rate': 0.0015073413747511038}. Best is trial 0 with value: 0.8572708368301392.
[I 2026-10-22 17:29:37,777] Trial 1 finished with value: 0.8579109311103821 and parameters: {'hidden1': 64, 'hidden2': 16, 'learning_rate': 0.0002217340099401778}. Best is trial 1 with value: 0.8579109311103821.
[I 2026-10-22 17:30:06,117] Trial 2 finished with value: 0.8604710698127747 and parameters: {'hidden1': 32, 'hidden2': 64, 'learning_rate': 0.0023745228627652932}. Best is trial 2 with value: 0.8604710698127747.
[I 2026-10-22 17:30:24,008] Trial 3 finished with value: 0.8591910004615784 and parameters: {'hidden1': 32, 'hidden2': 32, 'learning_rate': 0.00012268216498927657}. Best is trial 2 with value: 0.8604710698127747.
[I 2026-10-22 17:30:39,991] Trial 4 f

Best hyperparameters: {'hidden1': 32, 'hidden2': 64, 'learning_rate': 0.0023745228627652932}
Best validation accuracy: 0.8604710698127747


In [73]:
best_params = study.best_params

tuned_model = nn.Sequential(
    nn.Linear(105, best_params['hidden1']),
    nn.ReLU(),
    nn.Linear(best_params['hidden1'], best_params['hidden2']),
    nn.ReLU(),
    nn.Linear(best_params['hidden2'], 2)
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    tuned_model.parameters(),
    lr=best_params['learning_rate']
)

train_dataset = TensorDataset(X_train_opt, y_train_opt)
train_loader_opt = DataLoader(
    train_dataset, batch_size=32, shuffle=True
)

for epoch in range(10):
    tuned_model.train()

    for X_batch, y_batch in train_loader_opt:
        outputs = tuned_model(X_batch)
        loss = criterion(outputs, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch [{epoch+1}/10], Loss: {loss.item():.4f}")

Epoch [1/10], Loss: 0.2289
Epoch [2/10], Loss: 0.1304
Epoch [3/10], Loss: 0.2345
Epoch [4/10], Loss: 0.2418
Epoch [5/10], Loss: 0.1769
Epoch [6/10], Loss: 0.3356
Epoch [7/10], Loss: 0.3291
Epoch [8/10], Loss: 0.3013
Epoch [9/10], Loss: 0.3853
Epoch [10/10], Loss: 0.4789


In [74]:
tuned_model.eval()

with torch.no_grad():
    outputs = tuned_model(X_test_tensor)
    y_pred_tuned = torch.argmax(outputs, dim=1).numpy()

print("Accuracy:", accuracy_score(y_test_encoded, y_pred_tuned))
print("\nClassification Report:\n", classification_report(y_test_encoded, y_pred_tuned))
print("\nConfusion Matrix:\n", confusion_matrix(y_test_encoded, y_pred_tuned))

Accuracy: 0.858079049764489

Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.92      0.91      7428
           1       0.73      0.65      0.69      2338

    accuracy                           0.86      9766
   macro avg       0.81      0.79      0.80      9766
weighted avg       0.85      0.86      0.86      9766


Confusion Matrix:
 [[6865  563]
 [ 823 1515]]


In [75]:
results = pd.DataFrame({
    'Model': ['TensorFlow', 'PyTorch', 'Tuned PyTorch'],
    'Accuracy': [
        accuracy_score(y_test_encoded, y_pred),
        accuracy_score(y_test_encoded, y_pred_pytorch),
        accuracy_score(y_test_encoded, y_pred_tuned)
    ],
    'Precision': [
        precision_score(y_test_encoded, y_pred),
        precision_score(y_test_encoded, y_pred_pytorch),
        precision_score(y_test_encoded, y_pred_tuned)
    ],
    'Recall': [
        recall_score(y_test_encoded, y_pred),
        recall_score(y_test_encoded, y_pred_pytorch),
        recall_score(y_test_encoded, y_pred_tuned)
    ],
    'F1-Score': [
        f1_score(y_test_encoded, y_pred),
        f1_score(y_test_encoded, y_pred_pytorch),
        f1_score(y_test_encoded, y_pred_tuned)
    ]
})

results

,Model,Accuracy,Precision,Recall,F1-Score
0,TensorFlow,0.856134,0.749599,0.599230,0.666033
1,PyTorch,0.857157,0.748027,0.608212,0.670913
2,Tuned PyTorch,0.858079,0.729066,0.647990,0.686141


## Conclusion

TensorFlow and PyTorch models were implemented to predict income levels. The TensorFlow model achieved the highest accuracy (85.61%), while the Optuna-tuned PyTorch model achieved the highest F1-score (0.6861) and recall (64.80%), showing improved performance in identifying higher-income cases.